In [1]:
import sys
import os

path_to_scripts = os.path.join('..', '..', 'python_scripts')
sys.path.append(path_to_scripts)

%load_ext autoreload

In [5]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_error, max_error

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit

from data_to_bigquery import load_from_bigquery
from feature_engineering import drop_lag_nulls, validate_features, engineer_features
from baseline_model import baseline_model_xgb, xgb_train_preproc, evaluate_trained_model, pred_preproc_xgb, xgb_prediction

from statsmodels.graphics.tsaplots import plot_acf

%matplotlib inline

# Code to do split based on input date 

In [ ]:
# test train split for more years
# df = df.sort_values('datetime').copy()

# split_date = '2025-10-01'

# train_df = df[df['datetime'] < split_date]
# test_df = df[df['datetime'] >= split_date]

# X_train = train_df.drop(columns=['carbon_intensity', 'datetime'])
# y_train = train_df['carbon_intensity']

# X_test = test_df.drop(columns=['carbon_intensity', 'datetime'])
# y_test = test_df['carbon_intensity']

# Function checking 

In [ ]:
# raw table
df_raw = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
# display(df_raw.head())
# df_raw.columns

In [ ]:
# train preproc function
df_pre = xgb_train_preproc(df_raw, add_year_lag=True)
df_pre
# success working!

In [ ]:
# checking train_preproc
print(df_raw.shape)
print(df_pre.shape)
print(df_pre.columns.tolist())
print(df_pre.isna().sum().sum())
# success working!

In [ ]:
# checking model function
model, X_train, X_test, y_train, y_test = baseline_model_xgb()
# success working!

In [ ]:
# evaluate trained model
metrics = evaluate_trained_model(model, X_test, y_test)
print(metrics)

In [ ]:
# pred_preproc using X_test as df_new to check fucntion is working
#takes model ouput X_train
feature_cols = X_train.columns.tolist()

X_new = pred_preproc_xgb(df_new=X_test.copy(),
                     feature_cols=feature_cols
                     )

print(X_new.shape)
print(X_new.columns.tolist() == feature_cols)
print(X_new.isna().sum().sum())

In [ ]:
# checking prediction fucntion
prediction = xgb_prediction(
            model=model,
            df_new=X_test.copy(),
            feature_cols=feature_cols
            )

print(type(prediction))
print(len(prediction))
print(prediction[:5])

## CVGridsearch for params

# Splitting

In [ ]:
# loading data
df = load_from_bigquery('gridzero-489711', 'merged_set', 'test_merge_2017_onward_raw')
# df = xgb_train_preproc(df)

In [ ]:
# preproc
# defining X and target
# keeping dattime to help plotting
# splitting temporally bc multiple years

df = xgb_train_preproc(df)
target_col = 'carbon_intensity_gCO2_kWh'

# sort by datetime and reset index ooooo
df = df.sort_values('datetime').reset_index(drop=True)

target_col = 'carbon_intensity_gCO2_kWh'

# split option 1: by granular date
# split_date = '2024-01-01'
# train_df = df[df['datetime'] < split_date]
# test_df = df[df['datetime'] >= split_date]

# split option 2: by year
split_year = 2024
train_df = df[df['datetime'].dt.year < split_year]
test_df  = df[df['datetime'].dt.year >= split_year]

X_train = train_df.drop(columns=[target_col, 'datetime'])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col, 'datetime'])
y_test = test_df[target_col]

# keep only num cola to make xgboost happy
feature_cols = X_train.select_dtypes(include='number').columns.tolist()

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]


In [ ]:
# model split function

# def temporal_split(df, split_date, target):
#     df = df.sort_values('datetime')

#     train_df = df[df['datetime'] < split_date]
#     test_df  = df[df['datetime'] >= split_date]

#     X_train = train_df.drop(columns=[target, 'datetime'])
#     y_train = train_df[target]

#     X_test = test_df.drop(columns=[target, 'datetime'])
#     y_test = test_df[target]

#     return X_train, X_test, y_train, y_test

# Model tuning 

In [ ]:
# Baseline model from 2025 dataset

# Best params: {'learning_rate': np.float64(0.05), 'max_depth': np.int64(6), 'n_estimators': np.int64(1750)}
# MAE: 4.271311283111572
# RMSE: 5.715928011996834
# R2: 0.990020215511322
# Max Error: 35.22937774658203

# baseline_model = XGBRegressor(
#     learning_rate=0.05,
#     n_estimators=1750,
#     max_depth=6
#     )


In [ ]:
# Example XGBoost pipeline
#
# pipeline = pipeline = make_pipeline(
#         StandardScaler(),
#         XGBRegressor(
#             n_estimators=2000,
#             learning_rate=0.03,
#             max_depth=5,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             reg_alpha=0.1,
#             reg_lambda=1.0,
#             #histo
#             tree_method='hist',
#             random_state=42
#             )
#         )


In [8]:
# base model
xgb_stage1 = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# scoring metrics
scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2',
    'max_err':'neg_max_error'
}

# timeseries CV split
tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
# question ask Arthur about using tscv instead of cv
# from sklearn.model_selection import TimeSeriesSplit
# tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
#Hyperparams search

# stage 1 grid
param_grid_stage1 = {
    'n_estimators': [1250, 1500, 1750, 2000, 2250],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.03, 0.05, 0.07]
}

# grid search
grid_stage1 = GridSearchCV(
    estimator=xgb_stage1,
    param_grid=param_grid_stage1,
    scoring=scoring,
    refit='mae',
    cv=tscv,
    verbose=2,
    n_jobs=-1
)


In [ ]:
# stage 1 fit
grid_stage1.fit(X_train, y_train)


In [ ]:
# stage 1 best results
print('Stage 1 best params:', grid_stage1.best_params_)
print('Stage 1 best MAE:', -grid_stage1.best_score_)

In [ ]:
# stage 1 full results
stage1_results = pd.DataFrame(grid_stage1.cv_results_)[[
    'params',
    'mean_test_rmse',
    'mean_test_mae',
    'mean_test_r2',
    'mean_test_max_err']].copy() # stops SettingWithCopyWarning

stage1_results['mean_test_rmse'] = -stage1_results['mean_test_rmse']
stage1_results['mean_test_mae'] = -stage1_results['mean_test_mae']
stage1_results['mean_test_max_err'] = -stage1_results['mean_test_max_err']

stage1_results = stage1_results.sort_values('mean_test_mae')
stage1_results.head()

In [ ]:
# best params from stage 1
best_stage1_params = grid_stage1.best_params_

In [ ]:
# stage 2 model
xgb_stage2 = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    max_depth=best_stage1_params['max_depth']
)

In [ ]:
# stage 2 grid
param_grid_stage2 = {
    'learning_rate': [0.02, 0.03, 0.04],
    'n_estimators': [1450, 1500, 1550],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0]
}

In [ ]:
# stage 2 grid search
grid_stage2 = GridSearchCV(
    estimator=xgb_stage2,
    param_grid=param_grid_stage2,
    scoring=scoring,
    refit='mae',
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=False
)

In [ ]:
# stage 2 fit
grid_stage2.fit(X_train, y_train)

In [ ]:
# stage 2 best results
print('Stage 2 best params:', grid_stage2.best_params_)
print('Stage 2 best MAE:', -grid_stage2.best_score_)

In [ ]:
# stage 2 full results
stage2_results = pd.DataFrame(grid_stage2.cv_results_)[[
    'params',
    'mean_test_rmse',
    'mean_test_mae',
    'mean_test_r2',
    'mean_test_max_err']].copy() # stops SettingWithCopyWarning

stage2_results['mean_test_rmse'] = -stage2_results['mean_test_rmse']
stage2_results['mean_test_mae'] = -stage2_results['mean_test_mae']
stage2_results['mean_test_max_err'] = -stage2_results['mean_test_max_err']

stage2_results = stage2_results.sort_values('mean_test_mae')
stage2_results.head()

In [ ]:
# error ibvestigation
# predictions stage 1 and stage 2
y_pred_stage1 = grid_stage1.best_estimator_.predict(X_test)
y_pred_stage2 = grid_stage2.best_estimator_.predict(X_test)


In [ ]:
# table of errors
error_df = pd.DataFrame({
    'datetime': df.loc[X_test.index, 'datetime'],
    'y_true': y_test.values,
    'y_pred_s1': y_pred_stage1,
    'y_pred_s2': y_pred_stage2
})

error_df['abs_error_s1'] = (error_df['y_true'] - error_df['y_pred_s1']).abs()
error_df['abs_error_s2'] = (error_df['y_true'] - error_df['y_pred_s2']).abs()

In [ ]:
# table of worst errors to investigate
error_df
error_df.sort_values('abs_error_s2', ascending=False).head(20)

In [ ]:

error_df.sort_values('abs_error_s2', ascending=False).head(20)

In [ ]:
# Plot of worst errors
plt.figure(figsize=(14,4))
plt.plot(error_df['datetime'], error_df['abs_error_s1'], alpha=0.3)
plt.plot(error_df['datetime'], error_df['abs_error_s2'], alpha=0.8)
plt.title('Absolute error over time')
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(error_df['y_true'], error_df['abs_error_s1'], alpha=0.3)
plt.scatter(error_df['y_true'], error_df['abs_error_s2'], alpha=0.3)
plt.xlabel('True carbon intensity')
plt.ylabel('Absolute error')
plt.title('Error vs carbon intensity')
plt.show()

In [ ]:
print(pd.DataFrame(error_df['abs_error_s1'].describe()))

print(pd.DataFrame(error_df['abs_error_s2'].describe()))


In [ ]:
# plotting pred
plt.figure(figsize=(10,4))
plt.plot(df.loc[X_test.index, 'datetime'], y_test, label='Actual', linewidth=0.5)
plt.plot(df.loc[X_test.index, 'datetime'], y_pred_stage1, label='Predicted_S1', linewidth=1, alpha=0.3)
plt.plot(df.loc[X_test.index, 'datetime'], y_pred_stage2, label='Predicted_S2', linewidth=1, alpha=0.1)

plt.legend()
plt.title('Carbon intensity: Actual vs Predicted (Test Set)')
plt.xlabel('Time')
plt.ylabel('Carbon Intensity (gCO₂/kWh)')
plt.show()

In [ ]:
# Zoom in on worst period
mask = (df.loc[X_test.index, 'datetime'] > '2024-02-05') & \
       (df.loc[X_test.index, 'datetime'] < '2024-02-10')

plt.figure(figsize=(10,4))
plt.plot(df.loc[X_test.index, 'datetime'][mask], y_test[mask], label='Actual')
plt.plot(df.loc[X_test.index, 'datetime'][mask], y_pred[mask], label='Predicted')
plt.legend()
plt.ylabel('Carbon Intensity (gCO₂/kWh)')
plt.xlabel('Time')
plt.title('Magnified worst spike period')
plt.show()

In [ ]:
# finer stage 2 grid search with refit to mae and rmse to compare
# overnight run

#params grid
param_grid_overnight = {
    'n_estimators': [1400, 1500, 1600],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.02, 0.03, 0.04],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'gamma': [0, 1, 5]
}

#split
tscv = TimeSeriesSplit(n_splits=5)

# set scoring to inlcude
scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2',
    'max_err': 'neg_max_error'
}

# stage 2 finer
param_grid_overnight = {
    'n_estimators': [1450, 1500, 1550],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.02, 0.03, 0.04],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0]
}

xgb_base = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

# NOTE: The following MAE results not saved and overwritten 

In [ ]:
# refit to mae
# RESULTS NOT SAVED AND OVERWRITTEN
# re run as mae_correct lower down
grid_mae = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_overnight,
    scoring=scoring,
    refit='mae',
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=False # added bc do not need
)

grid_mae.fit(X_train, y_train)

print('Overnight MAE grid best params:', grid_mae.best_params_)
print('Overnight MAE grid best score:', -grid_mae.best_score_)

# NOTE: The following RMSE results have overwritten the previous MAE refit results 
grid_mae renamed to gird_rmse

In [ ]:
# refit to rmse
# NAMING ERROR - refit to rmse but named mae
# has overwritten the mae gridsearch
grid_mae = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_overnight,
    scoring=scoring,
    refit='rmse',
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=False # added bc do not need
)

grid_mae.fit(X_train, y_train)

print('Overnight MAE grid best params:', grid_mae.best_params_)
print('Overnight MAE grid best score:', -grid_mae.best_score_)

In [ ]:
# re naming the rmse grid and results
grid_rmse = grid_mae

print('Overnight RMSE grid best params:', grid_rmse.best_params_)
print('Overnight RMSE grid best score:', -grid_rmse.best_score_)

In [ ]:
# rmse results for overnight
rmse_results = pd.DataFrame(grid_rmse.cv_results_)[[
    'params',
    'mean_test_rmse',
    'mean_test_mae',
    'mean_test_r2',
    'mean_test_max_err'
]].copy() # nb remember to add this

rmse_results['mean_test_rmse'] = -rmse_results['mean_test_rmse']
rmse_results['mean_test_mae'] = -rmse_results['mean_test_mae']
rmse_results['mean_test_max_err'] = -rmse_results['mean_test_max_err']
rmse_results['mean_test_r2'] = -rmse_results['mean_test_r2']


rmse_results = rmse_results.sort_values('mean_test_rmse')
rmse_results.head(5)

# Rerun of MAE results beacuse overwritten above
saved as grid_mae_rerun

In [ ]:
# RE RUN
# CORRECT MAE
# refit to mae
grid_mae_rerun = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_overnight,
    scoring=scoring,
    refit='mae',
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=False # added bc do not need
)

grid_mae_rerun.fit(X_train, y_train)

print('Overnight MAE grid best params:', grid_mae_rerun.best_params_)
print('Overnight MAE grid best score:', -grid_mae_rerun.best_score_)

In [ ]:
# MAE RE RUN
# results for overnight
mae_rerun_results = pd.DataFrame(grid_mae_rerun.cv_results_)[[
    'params',
    'mean_test_rmse',
    'mean_test_mae',
    'mean_test_r2',
    'mean_test_max_err'
]].copy()

mae_rerun_results['mean_test_rmse'] *= -1 #= -mae_rerun_results['mean_test_rmse']
mae_rerun_results['mean_test_mae'] *= -1 #-mae_rerun_results['mean_test_mae']
mae_rerun_results['mean_test_max_err'] = -mae_rerun_results['mean_test_max_err']
mae_rerun_results['mean_test_r2'] = -mae_rerun_results['mean_test_r2']

mae_rerun_results = mae_rerun_results.sort_values('mean_test_mae')
mae_rerun_results.head(5)


In [ ]:
rmse_results.head(5)


In [ ]:
# stage 3 gridsearch
param_grid_stage3 = {
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 3, 5]
}

In [ ]:
# stage 3 mae gridsearch

best_mae_rerun_params = grid_mae_rerun.best_params_

xgb_stage3_mae = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    learning_rate=best_mae_rerun_params['learning_rate'],
    max_depth=best_mae_rerun_params['max_depth'],
    n_estimators=best_mae_rerun_params['n_estimators'],
    min_child_weight=best_mae_rerun_params['min_child_weight'],
    subsample=best_mae_rerun_params['subsample'],
    colsample_bytree=best_mae_rerun_params['colsample_bytree']
)

grid_stage3_mae = GridSearchCV(
    estimator=xgb_stage3_mae,
    param_grid=param_grid_stage3,
    scoring=scoring,
    refit='mae',
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=False
)

grid_stage3_mae.fit(X_train, y_train)

print('Stage 3 MAE best params:', grid_stage3_mae.best_params_)
print('Stage 3 MAE best score:', -grid_stage3_mae.best_score_)

In [ ]:
# stage 3 mae results
# saving best params
best_mae_s3_params = grid_stage3_mae.best_params_
best_mae_s3_scores = -grid_stage3_mae.best_score_
print('Stage 3 RMSE best params:', grid_stage3_mae.best_params_)
print('Stage 3 RMSE best score:', -grid_stage3_mae.best_score_)

stage3_mae_results = pd.DataFrame(grid_stage3_mae.cv_results_)[[
    'params',
    'mean_test_rmse',
    'mean_test_mae',
    'mean_test_r2',
    'mean_test_max_err'
]].copy()

stage3_mae_results['mean_test_rmse'] = -stage3_mae_results['mean_test_rmse']
stage3_mae_results['mean_test_mae'] = -stage3_mae_results['mean_test_mae']
stage3_mae_results['mean_test_max_err'] = -stage3_mae_results['mean_test_max_err']

stage3_mae_results = stage3_mae_results.sort_values('mean_test_mae')
stage3_mae_results.head(10)

In [ ]:
# stage 3 rmse gridsearch
best_rmse_params = grid_rmse.best_params_

xgb_stage3_rmse = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    learning_rate=best_rmse_params['learning_rate'],
    max_depth=best_rmse_params['max_depth'],
    n_estimators=best_rmse_params['n_estimators'],
    min_child_weight=best_rmse_params['min_child_weight'],
    subsample=best_rmse_params['subsample'],
    colsample_bytree=best_rmse_params['colsample_bytree']
)

grid_stage3_rmse = GridSearchCV(
    estimator=xgb_stage3_rmse,
    param_grid=param_grid_stage3,
    scoring=scoring,
    refit='rmse',
    cv=tscv,
    verbose=2,
    n_jobs=-1,
    return_train_score=False
)

grid_stage3_rmse.fit(X_train, y_train)

print('Stage 3 RMSE best params:', grid_stage3_rmse.best_params_)
print('Stage 3 RMSE best score:', -grid_stage3_rmse.best_score_)


In [ ]:
# stage 3 rmse results
# saving best params in variable
best_rmse_s3_params = grid_stage3_rmse.best_params_
best_rmse_s3_scores = -grid_stage3_rmse.best_score_
print('Stage 3 RMSE best params:', grid_stage3_rmse.best_params_)
print('Stage 3 RMSE best score:', -grid_stage3_rmse.best_score_)

#
stage3_rmse_results = pd.DataFrame(grid_stage3_rmse.cv_results_)[[
    'params',
    'mean_test_rmse',
    'mean_test_mae',
    'mean_test_r2',
    'mean_test_max_err'
]].copy()

stage3_rmse_results['mean_test_rmse'] = -stage3_rmse_results['mean_test_rmse']
stage3_rmse_results['mean_test_mae'] = -stage3_rmse_results['mean_test_mae']
stage3_rmse_results['mean_test_max_err'] = -stage3_rmse_results['mean_test_max_err']

stage3_rmse_results = stage3_rmse_results.sort_values('mean_test_rmse').copy()
stage1_results
stage3_rmse_results.head(10)


In [ ]:
stage3_mae_results.head(10)

In [ ]:
# stage 3 predictions
y_pred_final_mae_s3 = grid_stage3_mae.best_estimator_.predict(X_test)
y_pred_final_rmse_s3 = grid_stage3_rmse.best_estimator_.predict(X_test)

In [ ]:
# placeholder for plots needed

# plot 1: actual vs predicted
plt.figure(figsize=(10,4))
plt.scatter(y_test, y_pred_final_mae_s3, alpha=0.3)
plt.scatter(y_test, y_pred_final_rmse_s3, alpha=0.3)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red'
)

plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

In [ ]:
# plotting pred
plt.figure(figsize=(10,4))
plt.plot(df.loc[X_test.index, 'datetime'], y_test, label='Actual', linewidth=0.5)
plt.plot(df.loc[X_test.index, 'datetime'], y_pred_final_mae_s3, label='Predicted_S1', linewidth=1, alpha=0.4)
plt.plot(df.loc[X_test.index, 'datetime'], y_pred_final_rmse_s3, label='Predicted_S2', linewidth=1, alpha=0.3)

plt.legend()
plt.title('MAE VS RMSE Stage 3: Carbon intensity: Actual vs Predicted (Test Set)')
plt.xlabel('Time')
plt.ylabel('Carbon Intensity (gCO₂/kWh)')
plt.show()

In [ ]:
# Zoom in on worst period
mask = (df.loc[X_test.index, 'datetime'] > '2024-02-05') & \
       (df.loc[X_test.index, 'datetime'] < '2024-02-10')

plt.figure(figsize=(10,4))
plt.plot(df.loc[X_test.index, 'datetime'][mask], y_test[mask], label='Actual')
plt.plot(df.loc[X_test.index, 'datetime'][mask], y_pred_final_rmse_s3[mask], label='Predicted_RMSE')
plt.plot(df.loc[X_test.index, 'datetime'][mask], y_pred_final_mae_s3[mask], label='Predicted_MAE')
plt.legend()
plt.ylabel('Carbon Intensity (gCO₂/kWh)')
plt.xlabel('Time')
plt.title('Magnified worst spike period')
plt.show()

Note that rmse stage3 vs mae stage 3: 
- rmse refit has lower rmse, and mae
- rmse refit  has high r2
- rmse refit has higher mean max error by ~ 3
 Stage 3 optimised XGBoost model: 
 

In [ ]:
print(grid_rmse.best_params_)
print(best_rmse_s3_params)

model_s3_xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    learning_rate=best_rmse_params['learning_rate'],
    max_depth=best_rmse_params['max_depth'],
    n_estimators=best_rmse_params['n_estimators'],
    min_child_weight=best_rmse_params['min_child_weight'],
    subsample=best_rmse_params['subsample'],
    colsample_bytree=best_rmse_params['colsample_bytree'],
    gamma=best_rmse_s3_params['gamma'],
    reg_alpha=best_rmse_s3_params['reg_alpha'],
    reg_lambda=best_rmse_s3_params['reg_lambda']
)

# params hardcoded
    # random_state=42,
    # n_jobs=-1,
    # learning_rate=0.04,
    # max_depth=6
    # n_estimators=1450
    # min_child_weight=3
    # subsample=1.0,
    # colsample_bytree=0.7,
    # gamma=0,
    # reg_alpha=0.01,
    # reg_lambda=1


In [ ]:
# plot 2: timeseries line plot
plt.figure(figsize=(10,4))
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred, label='Predicted')

plt.legend()
plt.title('Carbon intensity over time')
plt.show()

In [ ]:
# plot 3: residual dist.
# residuals = y_test - y_pred

# plt.hist(residuals, bins=50)
# plt.title('Residual distribution')
# plt.show()

# Why timeseries even though xgboost: 
- xgboost is not a sequential model 
- the data is a timeseries
- so doing a timeseries split is useful 


# Autocorrelation check 

In [ ]:
df_raw

In [ ]:
df_pre.tail()

In [ ]:
# Autocorrelation
lags = len(df[target_col])
lags

plot_acf(df[target_col],lags=lags-1);

# Model on only last year new split needed

In [3]:
 # loading up to date df with cyclical feature eng
df = load_from_bigquery('gridzero-489711','merged_set','full_feature_engineered_data_test')


/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 148,991 rows and 31 columns from gridzero-489711.merged_set.full_feature_engineered_data_test


In [ ]:
df.head()
df.isna()

In [10]:
# preproc new splitting
# test on 2026
# train on 2025 ONLY
# defining X and target
# keeping dattime to help plotting
# still splitting temporally bc multiple years

target_col = 'carbon_intensity_gco2_kwh'

# sort by datetime and reset index ooooo
df = df.sort_values('datetime').reset_index(drop=True)

target_col = 'carbon_intensity_gco2_kwh'

# temporal split
train_df = df[df['datetime'].dt.year < 2025]
test_df  = df[df['datetime'].dt.year == 2025]

X_train = train_df.drop(columns=[target_col, 'datetime'])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col, 'datetime'])
y_test = test_df[target_col]

# keep only num cola to make xgboost happy
feature_cols = X_train.select_dtypes(include='number').columns.tolist()

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]

print(X_train.shape, X_test.shape)

(128064, 29) (17520, 29)


In [11]:
# new model and grid search
xgb_model = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    'n_estimators': [1500, 1750],
    'learning_rate': [0.03, 0.05, 0.07],
    'max_depth': [4, 6],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.7, 0.9],
    'gamma': [0, 0.1],
    'reg_alpha': [0, 0.01],
    'reg_lambda': [1, 3]
}

# # checkpoints
checkpoint_dir = './checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint = xgb.callback.TrainingCheckPoint(directory=checkpoint_dir)


grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring=scoring,
    refit='mae',
    cv=tscv,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

grid.fit(X_train, y_train, callbacks=[checkpoint])

print('Best params:', grid.best_params_)
print('Best CV MAE:', -grid.best_score_)

Fitting 5 folds for each of 768 candidates, totalling 3840 fits
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=1500, reg_alpha=0, reg_lambda=1, subsample=0.8; total time=   0.1s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=1500, reg_alpha=0, reg_lambda=3, subsample=0.8; total time=   0.0s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=1500, reg_alpha=0, reg_lambda=3, subsample=0.8; total time=   0.2s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=1500, reg_alpha=0, reg_lambda=3, subsample=0.8; total time=   0.0s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.03, max_depth=4, min_child_weight=1, n_estimators=1500, reg_alpha=0, reg_lambda=3, subsample=0.8; total time=   0.1s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.03, max_depth=4

ValueError: 
All the 3840 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3840 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
TypeError: XGBModel.fit() got an unexpected keyword argument 'callbacks'


In [ ]:
# results
best_params = grid.best_params_

best_xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    **best_params
)

best_xgb.fit(X_train, y_train)

y_pred = best_xgb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
mxerr = max_error(y_test, y_pred)

print('Test MAE:', mae)
print('Test RMSE:', rmse)
print('Test R2:', r2)
print('Test Max Error:', mxerr)
